In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/ground_truth_coordinates.csv
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_04498.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_16279.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_11219.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_00610.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_12470.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_13700.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_10731.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_06407.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_03973.jpg
/kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images/IMG_12577.jpg
/kaggle/input/datasets/yadhunandand

In [2]:
!pip install transformers scikit-learn tqdm -q

In [3]:
%%writefile 1_extract_embeddings.py
"""
Step 1: Extract frozen CLIP image embeddings for your whole dataset.
Runs once. GPU-heavy step.
"""

import argparse
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--csv", required=True)
    p.add_argument("--image_dir", required=True)
    p.add_argument("--out_dir", default="embeddings")
    p.add_argument("--model_name", default="openai/clip-vit-base-patch32")
    p.add_argument("--batch_size", type=int, default=64)
    p.add_argument("--id_col", default="image_id")
    p.add_argument("--lat_col", default="latitude")
    p.add_argument("--lon_col", default="longitude")
    p.add_argument("--image_ext", default=".jpg")
    return p.parse_args()


def resolve_image_path(image_dir, image_id, default_ext):
    known_exts = (".jpg", ".jpeg", ".png", ".webp")
    if str(image_id).lower().endswith(known_exts):
        return os.path.join(image_dir, str(image_id))
    return os.path.join(image_dir, f"{image_id}{default_ext}")


def extract_feature_tensor(output):
    """Handles different transformers versions returning different formats
    from get_image_features."""
    if torch.is_tensor(output):
        return output
    if hasattr(output, "image_embeds"):
        return output.image_embeds
    if hasattr(output, "pooler_output"):
        return output.pooler_output
    raise TypeError(f"Unexpected output type from get_image_features: {type(output)}")


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    df = pd.read_csv(args.csv)
    assert {args.id_col, args.lat_col, args.lon_col}.issubset(df.columns), \
        f"CSV must have {args.id_col}, {args.lat_col}, {args.lon_col} columns. Found: {list(df.columns)}"
    print(f"Loaded {len(df)} rows from {args.csv}")

    print(f"Loading frozen encoder: {args.model_name}")
    model = CLIPModel.from_pretrained(args.model_name).to(device)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    processor = CLIPProcessor.from_pretrained(args.model_name)

    all_embeddings = []
    all_coords = []
    all_paths = []
    failed = []

    batch_imgs, batch_coords, batch_paths = [], [], []

    def flush_batch():
        if not batch_imgs:
            return
        inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            raw_output = model.get_image_features(**inputs)
            feats = extract_feature_tensor(raw_output)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeddings.append(feats.cpu().numpy())
        all_coords.extend(batch_coords)
        all_paths.extend(batch_paths)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting embeddings"):
        img_path = resolve_image_path(args.image_dir, row[args.id_col], args.image_ext)
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            failed.append((img_path, str(e)))
            continue

        batch_imgs.append(img)
        batch_coords.append((row[args.lat_col], row[args.lon_col]))
        batch_paths.append(img_path)

        if len(batch_imgs) >= args.batch_size:
            flush_batch()
            batch_imgs, batch_coords, batch_paths = [], [], []

    flush_batch()

    embeddings = np.concatenate(all_embeddings, axis=0)
    coords = np.array(all_coords, dtype=np.float64)

    np.save(os.path.join(args.out_dir, "embeddings.npy"), embeddings)
    np.save(os.path.join(args.out_dir, "coords.npy"), coords)
    with open(os.path.join(args.out_dir, "paths.txt"), "w") as f:
        f.write("\n".join(all_paths))

    print(f"\nDone. Saved {embeddings.shape[0]} embeddings of dim {embeddings.shape[1]}")
    print(f"  -> {args.out_dir}/embeddings.npy")
    print(f"  -> {args.out_dir}/coords.npy")
    if failed:
        print(f"\n{len(failed)} images failed to load (skipped). First few:")
        for p, e in failed[:5]:
            print(f"  {p}: {e}")


if __name__ == "__main__":
    main()

Writing 1_extract_embeddings.py


In [4]:
%%writefile 2_train.py
"""
Step 2 (v4): Build location clusters, train a 3-output head:
  1. classification -- which coarse cluster
  2. regression -- fine (lat, lon) offset within that cluster
  3. radius -- a predicted confidence radius (km), now informed by the
     classifier's own confidence (max softmax prob + entropy), so it can
     actually vary per-example instead of collapsing to one flat number.

Then calibrate the radius PER PREDICTED CLUSTER (with a fallback to a
global multiplier for clusters with too few validation examples), so
easy/confident clusters keep tight radii instead of everyone getting
the same bloated "safe" radius.
"""

import argparse
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

EARTH_RADIUS_KM = 6371.0
MIN_RADIUS_KM = 1.0
MAX_RADIUS_KM = 20000.0
MIN_EXAMPLES_FOR_PERCLUSTER_CALIB = 15  # below this, fall back to global multiplier


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--emb_dir", default="embeddings")
    p.add_argument("--out_dir", default="head_model")
    p.add_argument("--n_clusters", type=int, default=100)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--reg_weight", type=float, default=1.0)
    p.add_argument("--radius_weight", type=float, default=2.0)
    p.add_argument("--val_frac", type=float, default=0.15)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--calibration_percentile", type=float, default=70.0)
    return p.parse_args()


def latlon_to_unit_xyz(latlon_deg):
    lat = np.radians(latlon_deg[:, 0])
    lon = np.radians(latlon_deg[:, 1])
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.stack([x, y, z], axis=1)


def unit_xyz_to_latlon(xyz):
    xyz = xyz / np.linalg.norm(xyz, axis=1, keepdims=True)
    lat = np.degrees(np.arcsin(np.clip(xyz[:, 2], -1, 1)))
    lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
    return np.stack([lat, lon], axis=1)


def haversine_km(latlon1, latlon2):
    lat1, lon1 = torch.deg2rad(latlon1[:, 0]), torch.deg2rad(latlon1[:, 1])
    lat2, lon2 = torch.deg2rad(latlon2[:, 0]), torch.deg2rad(latlon2[:, 1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    c = 2 * torch.arcsin(torch.sqrt(torch.clamp(a, 0, 1)))
    return EARTH_RADIUS_KM * c


class GeoHead(nn.Module):
    """
    Three outputs: cluster classification, fine (lat,lon) offset, and a
    predicted confidence radius (km). The radius head additionally sees
    the classifier's own confidence signals (max softmax prob + entropy)
    so it can learn to output a SMALL radius when the model is sure which
    region it's in, and a LARGE radius when it's not -- instead of
    collapsing to one average number regardless of how hard the image is.
    """

    def __init__(self, embed_dim, n_clusters, hidden=512):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(hidden, n_clusters)
        self.regressor = nn.Linear(hidden, 2)
        # radius head takes backbone features PLUS 2 confidence scalars
        self.radius_head = nn.Sequential(
            nn.Linear(hidden + 2, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        h = self.backbone(x)
        logits = self.classifier(h)
        offset = self.regressor(h)

        probs = F.softmax(logits, dim=-1)
        max_prob = probs.max(dim=-1, keepdim=True).values
        entropy = -(probs * torch.log(probs.clamp(min=1e-8))).sum(dim=-1, keepdim=True)
        # normalize entropy roughly to [0,1]-ish range using log(n_clusters)
        entropy = entropy / np.log(logits.shape[-1])

        radius_input = torch.cat([h, max_prob, entropy], dim=-1)
        log_radius = self.radius_head(radius_input)
        return logits, offset, log_radius


class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, cluster_labels, residuals):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.cluster_labels = torch.tensor(cluster_labels, dtype=torch.long)
        self.residuals = torch.tensor(residuals, dtype=torch.float32)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.cluster_labels[idx], self.residuals[idx]


def main():
    args = parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    embeddings = np.load(os.path.join(args.emb_dir, "embeddings.npy"))
    coords = np.load(os.path.join(args.emb_dir, "coords.npy"))
    print(f"Loaded {embeddings.shape[0]} embeddings, dim={embeddings.shape[1]}")

    print(f"Clustering into {args.n_clusters} location cells (spherical KMeans)...")
    xyz = latlon_to_unit_xyz(coords)
    kmeans = KMeans(n_clusters=args.n_clusters, random_state=args.seed, n_init=10)
    cluster_labels = kmeans.fit_predict(xyz)
    cluster_centers_latlon = unit_xyz_to_latlon(kmeans.cluster_centers_)

    counts = np.bincount(cluster_labels, minlength=args.n_clusters)
    tiny = (counts < 5).sum()
    if tiny > 0:
        print(f"  Warning: {tiny} clusters have < 5 images. Consider lowering --n_clusters.")
    print(f"  Cluster sizes: min={counts.min()}, median={int(np.median(counts))}, max={counts.max()}")

    residuals = coords - cluster_centers_latlon[cluster_labels]

    idx_train, idx_val = train_test_split(
        np.arange(len(embeddings)), test_size=args.val_frac,
        random_state=args.seed, stratify=cluster_labels
        if counts.min() >= 2 else None,
    )

    train_ds = EmbeddingDataset(embeddings[idx_train], cluster_labels[idx_train], residuals[idx_train])
    val_ds = EmbeddingDataset(embeddings[idx_val], cluster_labels[idx_val], residuals[idx_val])
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False)

    model = GeoHead(embed_dim=embeddings.shape[1], n_clusters=args.n_clusters).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()

    centers_t = torch.tensor(cluster_centers_latlon, dtype=torch.float32).to(device)

    best_val_km = float("inf")
    best_state = None

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss_total = 0.0
        for emb, cls, res in train_loader:
            emb, cls, res = emb.to(device), cls.to(device), res.to(device)
            optimizer.zero_grad()
            logits, pred_res, log_radius = model(emb)

            loss_cls = ce_loss(logits, cls)
            loss_reg = mse_loss(pred_res, res)

            pred_latlon = centers_t[cls] + pred_res
            true_latlon = centers_t[cls] + res
            actual_error_km = haversine_km(pred_latlon, true_latlon).detach()
            log_target = torch.log(actual_error_km.clamp(min=MIN_RADIUS_KM) + 1.0)
            loss_radius = mse_loss(log_radius.squeeze(-1), log_target)

            loss = loss_cls + args.reg_weight * loss_reg + args.radius_weight * loss_radius
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item() * emb.size(0)
        train_loss = train_loss_total / len(train_ds)

        model.eval()
        val_errors_km = []
        val_radii_km = []
        correct_cluster = 0
        covered = 0
        with torch.no_grad():
            for emb, cls, res in val_loader:
                emb, cls, res = emb.to(device), cls.to(device), res.to(device)
                logits, pred_res, log_radius = model(emb)
                pred_cluster = logits.argmax(dim=1)
                correct_cluster += (pred_cluster == cls).sum().item()

                pred_latlon = centers_t[pred_cluster] + pred_res
                true_latlon = centers_t[cls] + res
                errs = haversine_km(pred_latlon, true_latlon)
                val_errors_km.extend(errs.cpu().numpy().tolist())

                pred_radius = torch.exp(log_radius.squeeze(-1)).clamp(MIN_RADIUS_KM, MAX_RADIUS_KM)
                val_radii_km.extend(pred_radius.cpu().numpy().tolist())
                covered += (errs <= pred_radius).sum().item()

        val_km = float(np.median(val_errors_km))
        cluster_acc = correct_cluster / len(val_ds)
        avg_radius = float(np.mean(val_radii_km))
        std_radius = float(np.std(val_radii_km))
        coverage = covered / len(val_ds)
        print(f"Epoch {epoch:3d} | train_loss {train_loss:.4f} | "
              f"val median error {val_km:8.1f} km | cluster acc {cluster_acc:.3f} | "
              f"avg radius {avg_radius:7.1f} km (std {std_radius:6.1f}) | coverage {coverage:.3f}")

        if val_km < best_val_km:
            best_val_km = val_km
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\nBest val median error: {best_val_km:.1f} km")

    # ---- Per-cluster calibration ----
    print("\nCalibrating radius PER-CLUSTER on validation set using best checkpoint...")
    model.load_state_dict(best_state)
    model.eval()

    all_errs, all_radii, all_pred_clusters = [], [], []
    with torch.no_grad():
        for emb, cls, res in val_loader:
            emb, cls, res = emb.to(device), cls.to(device), res.to(device)
            logits, pred_res, log_radius = model(emb)
            pred_cluster = logits.argmax(dim=1)
            pred_latlon = centers_t[pred_cluster] + pred_res
            true_latlon = centers_t[cls] + res
            errs = haversine_km(pred_latlon, true_latlon)
            radius = torch.exp(log_radius.squeeze(-1)).clamp(MIN_RADIUS_KM, MAX_RADIUS_KM)
            all_errs.extend(errs.cpu().numpy().tolist())
            all_radii.extend(radius.cpu().numpy().tolist())
            all_pred_clusters.extend(pred_cluster.cpu().numpy().tolist())

    all_errs = np.array(all_errs)
    all_radii = np.array(all_radii)
    all_pred_clusters = np.array(all_pred_clusters)

    # global fallback multiplier (same as before)
    global_ratios = all_errs / np.clip(all_radii, MIN_RADIUS_KM, None)
    global_multiplier = max(float(np.percentile(global_ratios, args.calibration_percentile)), 1.0)

    # per predicted-cluster multiplier, falling back to global when sparse
    cluster_multipliers = np.full(args.n_clusters, global_multiplier, dtype=np.float64)
    n_calibrated_directly = 0
    for c in range(args.n_clusters):
        mask = all_pred_clusters == c
        if mask.sum() >= MIN_EXAMPLES_FOR_PERCLUSTER_CALIB:
            ratios_c = all_errs[mask] / np.clip(all_radii[mask], MIN_RADIUS_KM, None)
            mult_c = max(float(np.percentile(ratios_c, args.calibration_percentile)), 1.0)
            cluster_multipliers[c] = mult_c
            n_calibrated_directly += 1

    calibrated_radii = all_radii * cluster_multipliers[all_pred_clusters]
    calibrated_coverage = float((all_errs <= calibrated_radii).mean())

    print(f"  Clusters calibrated directly (>= {MIN_EXAMPLES_FOR_PERCLUSTER_CALIB} val examples): "
          f"{n_calibrated_directly}/{args.n_clusters} (rest use global fallback {global_multiplier:.2f}x)")
    print(f"  Coverage BEFORE calibration: {float((all_errs <= all_radii).mean()):.3f}")
    print(f"  Coverage AFTER  calibration: {calibrated_coverage:.3f}")
    print(f"  Avg radius BEFORE: {all_radii.mean():.1f} km | AFTER: {calibrated_radii.mean():.1f} km "
          f"(std {calibrated_radii.std():.1f})")

    torch.save({
        "model_state": best_state,
        "embed_dim": embeddings.shape[1],
        "n_clusters": args.n_clusters,
        "cluster_centers_latlon": cluster_centers_latlon,
        "cluster_calibration_multipliers": cluster_multipliers,  # array, size n_clusters
        "global_calibration_multiplier": global_multiplier,       # scalar fallback
    }, os.path.join(args.out_dir, "best_head.pt"))

    print(f"\nSaved calibrated model -> {args.out_dir}/best_head.pt")


if __name__ == "__main__":
    main()

Writing 2_train.py


In [5]:
%%writefile 3_infer.py
"""
Step 3: Inference on a single new image. Fully offline.
"""

import argparse
import torch
import torch.nn as nn
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


class GeoHead(nn.Module):
    def __init__(self, embed_dim, n_clusters, hidden=512):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(hidden, n_clusters)
        self.regressor = nn.Linear(hidden, 2)

    def forward(self, x):
        h = self.backbone(x)
        return self.classifier(h), self.regressor(h)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--image", required=True)
    p.add_argument("--head_dir", default="head_model")
    p.add_argument("--model_name", default="openai/clip-vit-base-patch32")
    return p.parse_args()


def main():
    args = parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    encoder = CLIPModel.from_pretrained(args.model_name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(args.model_name)

    ckpt = torch.load(f"{args.head_dir}/best_head.pt", map_location=device)
    head = GeoHead(ckpt["embed_dim"], ckpt["n_clusters"]).to(device)
    head.load_state_dict(ckpt["model_state"])
    head.eval()
    centers = torch.tensor(ckpt["cluster_centers_latlon"], dtype=torch.float32).to(device)

    img = Image.open(args.image).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        emb = encoder.get_image_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        logits, pred_res = head(emb)
        pred_cluster = logits.argmax(dim=1)
        pred_latlon = (centers[pred_cluster] + pred_res)[0].cpu().numpy()

        probs = torch.softmax(logits, dim=1)[0]
        top3 = torch.topk(probs, k=3)

    print(f"Predicted location: lat={pred_latlon[0]:.4f}, lon={pred_latlon[1]:.4f}")
    print("\nTop-3 candidate regions (cluster centroid, confidence):")
    for rank, (p, idx) in enumerate(zip(top3.values, top3.indices), 1):
        c = centers[idx].cpu().numpy()
        print(f"  {rank}. lat={c[0]:.2f}, lon={c[1]:.2f}  ({p.item()*100:.1f}%)")


if __name__ == "__main__":
    main()

Writing 3_infer.py


In [6]:
!python 1_extract_embeddings.py \
    --csv /kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/ground_truth_coordinates.csv \
    --image_dir /kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/images \
    --out_dir /kaggle/working/embeddings

Using device: cuda
Loaded 19002 rows from /kaggle/input/datasets/yadhunandandme26b146/mydata/noised_dataset/ground_truth_coordinates.csv
Loading frozen encoder: openai/clip-vit-base-patch32
config.json: 4.19kB [00:00, 10.3MB/s]
pytorch_model.bin: 100%|██████████████████████| 605M/605M [00:03<00:00, 159MB/s]
model.safetensors:   0%|                             | 0.00/605M [00:00<?, ?B/s]
Loading weights:   0%|                                  | 0/398 [00:00<?, ?it/s]
Loading weights:   0%| | 1/398 [00:00<00:00, 8943.08it/s, Materializing param=lo
Loading weights:   0%| | 1/398 [00:00<00:00, 684.23it/s, Materializing param=log
Loading weights:   1%| | 2/398 [00:00<00:00, 964.54it/s, Materializing param=tex
Loading weights:   1%| | 2/398 [00:00<00:00, 619.54it/s, Materializing param=tex
Loading weights:   1%| | 3/398 [00:00<00:00, 408.88it/s, Materializing param=tex
Loading weights:   1%| | 3/398 [00:00<00:01, 359.67it/s, Materializing param=tex
Loading weights:   1%| | 4/398 [00:00<00:00

In [7]:
!python 2_train.py --emb_dir /kaggle/working/embeddings --out_dir /kaggle/working/head_model --epochs 30

Using device: cuda
Loaded 19002 embeddings, dim=512
Clustering into 100 location cells (spherical KMeans)...
  Cluster sizes: min=13, median=192, max=537
Epoch   1 | train_loss 27.6940 | val median error   9084.1 km | cluster acc 0.018 | avg radius   413.6 km (std  141.3) | coverage 0.014
Epoch   2 | train_loss 17.2214 | val median error   9386.6 km | cluster acc 0.028 | avg radius   380.9 km (std   96.5) | coverage 0.023
Epoch   3 | train_loss 17.1465 | val median error   9369.2 km | cluster acc 0.028 | avg radius   353.5 km (std   88.4) | coverage 0.021
Epoch   4 | train_loss 17.1116 | val median error   8076.3 km | cluster acc 0.034 | avg radius   346.5 km (std   84.1) | coverage 0.021
Epoch   5 | train_loss 17.0063 | val median error   8675.9 km | cluster acc 0.042 | avg radius   289.7 km (std   68.2) | coverage 0.022
Epoch   6 | train_loss 16.8265 | val median error   8092.9 km | cluster acc 0.049 | avg radius   329.9 km (std   69.9) | coverage 0.026
Epoch   7 | train_loss 16.5952

In [8]:
%%writefile 0_save_clip_locally.py
"""
Run once. Downloads CLIP and saves it to a local folder, so inference
scripts can load it from disk with zero network access -- required by
the hackathon's offline-inference rule (Section 4.3).
"""

from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
SAVE_DIR = "/kaggle/working/clip_local"

print(f"Downloading {MODEL_NAME}...")
model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f"Saved locally to {SAVE_DIR}")
print("This folder must be bundled with your final submission (e.g. as a Kaggle Dataset/Output) so inference never needs internet.")

Writing 0_save_clip_locally.py


In [9]:
!python 0_save_clip_locally.py

Loading weights: 100%|█| 398/398 [00:00<00:00, 2708.84it/s, Materializing param=
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.11it/s]
Saved locally to /kaggle/working/clip_local
This folder must be bundled with your final submission (e.g. as a Kaggle Dataset

In [10]:
!python 2_train.py --emb_dir /kaggle/working/embeddings --out_dir /kaggle/working/head_model --n_clusters 100 --epochs 30

Using device: cuda
Loaded 19002 embeddings, dim=512
Clustering into 100 location cells (spherical KMeans)...
  Cluster sizes: min=13, median=192, max=537
Epoch   1 | train_loss 27.6940 | val median error   9084.1 km | cluster acc 0.018 | avg radius   413.6 km (std  141.3) | coverage 0.014
Epoch   2 | train_loss 17.2214 | val median error   9386.6 km | cluster acc 0.028 | avg radius   380.9 km (std   96.5) | coverage 0.023
Epoch   3 | train_loss 17.1465 | val median error   9369.2 km | cluster acc 0.028 | avg radius   353.5 km (std   88.4) | coverage 0.021
Epoch   4 | train_loss 17.1116 | val median error   8076.3 km | cluster acc 0.034 | avg radius   346.5 km (std   84.1) | coverage 0.021
Epoch   5 | train_loss 17.0063 | val median error   8675.9 km | cluster acc 0.042 | avg radius   289.7 km (std   68.2) | coverage 0.022
Epoch   6 | train_loss 16.8265 | val median error   8092.9 km | cluster acc 0.049 | avg radius   329.9 km (std   69.9) | coverage 0.026
Epoch   7 | train_loss 16.5952